# 🛠️ VoltVision — Feature Engineering

EDA gave us a pretty clear picture of the dataset. Now we can start changing the data — but only where the earlier analysis gave us a reason to do so. ⚡

In this notebook we will keep the process **clean and focused**:

**raw train/test → remove ID from model features → encode meaningful categories → create a small set of justified features → one-hot encode nominal columns → verify train/test alignment → save processed data**

A few decisions are already settled from EDA:

- there are **no missing values**, so we do not need imputation;
- there are **no duplicate rows**, so we do not need duplicate removal;
- `id` behaves like a row identifier, so it should not be used as a predictive feature;
- `Range_Anxiety_Level` has a clear order, so ordinal encoding makes sense;
- subsidy, environmental concern, home charging, income, and range anxiety showed the strongest relationships with EV purchase;
- the Kaggle test set is the real hidden test set, so we will **not split it here**.

This notebook will stay **basic-to-medium and readable** — no SMOTE, target encoding, feature selection, Optuna, or model-based tricks yet.

## Imports and Data Loading

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [2]:
test.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low


In [3]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (668665, 15)
Test shape: (286571, 14)


Okay, we are starting again from the **raw competition files**. ✅

That is intentional. The EDA notebook was only for investigation, while every real transformation starts fresh here.

The training data has **668,665 rows and 15 columns**, while the Kaggle test data has **286,571 rows and 14 columns**. The only extra training column is our target, `Will_Buy_EV`.

## Target and ID Separation

In [4]:
train_ids = train["id"].copy()
test_ids = test["id"].copy()

y = train["Will_Buy_EV"].map({"No":0 , "Yes":1})

X_train = train.drop(columns=["id" , "Will_Buy_EV"]).copy()
X_test = test.drop(columns="id").copy()

X_train.shape , X_test.shape , y.shape

((668665, 13), (286571, 13), (668665,))

In [5]:
print("Target values:")
print(y.value_counts().sort_index())

print("\nPositive class rate:", round(y.mean() * 100 , 2), "%")

Target values:
Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64

Positive class rate: 17.46 %


Great — the target and identifiers are now separated. 🎯

`Will_Buy_EV` is encoded as:

- `0` → No
- `1` → Yes

We also kept the original train/test IDs separately for traceability and Kaggle submission work later, but `id` itself will **not enter the model features**.

The positive class is still **17.46%**, so Notebook 03 should use stratified validation and treat ROC-AUC as the main metric.

## Binary and Ordinal Encoding

In [6]:
binary_cols = ["Home_Charging_Possible" , "Subsidy_Available"]

for col in binary_cols:
    X_train[col] = X_train[col].map({"No":0 , "Yes":1})
    X_test[col] = X_test[col].map({"No":0 , "Yes":1})

X_train[binary_cols].head()

,Home_Charging_Possible,Subsidy_Available
0,1,0
1,1,0
2,0,1
3,1,0
4,1,0


In [7]:
range_map = {"Low":0 , "Medium":1 , "High":2}

X_train["Range_Anxiety_Level"] = X_train["Range_Anxiety_Level"].map(range_map)
X_test["Range_Anxiety_Level"] = X_test["Range_Anxiety_Level"].map(range_map)

X_train["Environmental_Concern_Level"] = X_train["Environmental_Concern_Level"].astype(int)
X_test["Environmental_Concern_Level"] = X_test["Environmental_Concern_Level"].astype(int)

X_train[["Range_Anxiety_Level" , "Environmental_Concern_Level"]].head()

,Range_Anxiety_Level,Environmental_Concern_Level
0,0,1
1,0,4
2,0,5
3,0,3
4,0,3


In [8]:
print("Home charging values:", sorted(X_train["Home_Charging_Possible"].unique()))
print("Subsidy values:", sorted(X_train["Subsidy_Available"].unique()))
print("Range anxiety values:", sorted(X_train["Range_Anxiety_Level"].unique()))
print("Concern levels:", sorted(X_train["Environmental_Concern_Level"].unique()))

Home charging values: [np.int64(0), np.int64(1)]
Subsidy values: [np.int64(0), np.int64(1)]
Range anxiety values: [np.int64(0), np.int64(1), np.int64(2)]
Concern levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


Nice — the meaningful ordered/binary variables are now represented numerically without losing their structure. ✅

`Range_Anxiety_Level` is especially important here. EDA showed a very clear progression:

**Low anxiety → much higher EV purchase rate → Medium → High**

So treating it as an ordered `0 → 1 → 2` variable is more informative than assigning arbitrary category numbers.

`Environmental_Concern_Level` was already an ordered 1–5 scale, so we simply keep that structure as integer values.

## Charging Infrastructure Features

In [9]:
for df in [X_train , X_test]:
    df["Total_Charging_Stations"] = (
        df["Charging_Stations_Near_Home"] +
        df["Charging_Stations_Near_Work"]
    )

X_train[[
    "Charging_Stations_Near_Home" ,
    "Charging_Stations_Near_Work" ,
    "Total_Charging_Stations"
]].head()

,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Total_Charging_Stations
0,3,7,10
1,2,2,4
2,8,15,23
3,6,9,15
4,2,3,5


In [10]:
X_train["Total_Charging_Stations"].describe()

count    668665.000000
mean         12.136722
std           7.946181
min           0.000000
25%           5.000000
50%          11.000000
75%          18.000000
max          33.000000
Name: Total_Charging_Stations, dtype: float64

This is a simple but useful infrastructure feature.

Instead of making the model reconstruct total nearby public charging availability from two separate columns every time, `Total_Charging_Stations` gives one direct summary of the customer's surrounding charging network.

We are **not** creating lots of arbitrary charging ratios here. EDA showed that home charging itself is more meaningful than blindly combining every charging variable, so we keep this part intentionally small.

## Affordability and Ownership Features

In [11]:
for df in [X_train , X_test]:
    df["Income_Per_Car"] = (
        df["Annual_Income_USD"] /
        df["Number_of_Cars_Owned"].clip(lower=1)
    )

X_train[[
    "Annual_Income_USD" ,
    "Number_of_Cars_Owned" ,
    "Income_Per_Car"
]].head()

,Annual_Income_USD,Number_of_Cars_Owned,Income_Per_Car
0,92887.0,2,46443.5
1,30000.0,1,30000.0
2,94389.0,1,94389.0
3,73580.0,2,36790.0
4,57898.0,1,57898.0


In [12]:
X_train["Income_Per_Car"].describe()

count    668665.000000
mean      58754.583977
std       31488.101505
min        7500.000000
25%       33897.000000
50%       50559.500000
75%       81082.000000
max      188549.000000
Name: Income_Per_Car, dtype: float64

EDA showed a strong positive relationship between annual income and EV purchase.

`Income_Per_Car` adds a simple affordability perspective: two customers can earn the same amount, but one may already support several vehicles while another owns none or one.

For customers with zero cars we use a minimum denominator of `1`, which avoids division by zero without creating artificial missing values.

## Interaction Feature Creation

In [13]:
for df in [X_train , X_test]:
    df["Low_Range_Anxiety"] = (df["Range_Anxiety_Level"] == 0).astype(int)

    df["Subsidy_x_Concern"] = (
        df["Subsidy_Available"] *
        df["Environmental_Concern_Level"]
    )

    df["Subsidy_x_LowAnxiety"] = (
        df["Subsidy_Available"] *
        df["Low_Range_Anxiety"]
    )

    df["Subsidy_x_HomeCharging"] = (
        df["Subsidy_Available"] *
        df["Home_Charging_Possible"]
    )

    df["Income_x_Concern"] = (
        (df["Annual_Income_USD"] / 10000) *
        df["Environmental_Concern_Level"]
    )

    df["Subsidy_x_Income"] = (
        df["Subsidy_Available"] *
        (df["Annual_Income_USD"] / 10000)
    )

engineered_cols = [
    "Total_Charging_Stations" ,
    "Income_Per_Car" ,
    "Low_Range_Anxiety" ,
    "Subsidy_x_Concern" ,
    "Subsidy_x_LowAnxiety" ,
    "Subsidy_x_HomeCharging" ,
    "Income_x_Concern" ,
    "Subsidy_x_Income"
]

X_train[engineered_cols].head()

,Total_Charging_Stations,Income_Per_Car,Low_Range_Anxiety,Subsidy_x_Concern,Subsidy_x_LowAnxiety,Subsidy_x_HomeCharging,Income_x_Concern,Subsidy_x_Income
0,10,46443.5,1,0,0,0,9.2887,0.0000
1,4,30000.0,1,0,0,0,12.0000,0.0000
2,23,94389.0,1,5,1,0,47.1945,9.4389
3,15,36790.0,1,0,0,0,22.0740,0.0000
4,5,57898.0,1,0,0,0,17.3694,0.0000


In [14]:
X_train[engineered_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Total_Charging_Stations,668665.0,12.14,7.95,0.0,5.00,11.00,18.00,33.00
Income_Per_Car,668665.0,58754.58,31488.10,7500.0,33897.00,50559.50,81082.00,188549.00
Low_Range_Anxiety,668665.0,0.90,0.30,0.0,1.00,1.00,1.00,1.00
Subsidy_x_Concern,668665.0,1.90,1.86,0.0,0.00,1.00,4.00,5.00
Subsidy_x_LowAnxiety,668665.0,0.57,0.50,0.0,0.00,1.00,1.00,1.00
Subsidy_x_HomeCharging,668665.0,0.44,0.50,0.0,0.00,0.00,1.00,1.00
Income_x_Concern,668665.0,25.20,15.99,3.0,11.79,22.19,36.19,94.27
Subsidy_x_Income,668665.0,5.39,4.75,0.0,0.00,6.37,9.27,18.85


Ohh, this is the most important feature-engineering step. 🔎

We created only a **small number of interactions that EDA actually justified**:

- `Subsidy_x_Concern` captures the very strong subsidy + environmental-motivation pattern.
- `Subsidy_x_LowAnxiety` captures the fact that subsidy works best when range anxiety is already low.
- `Subsidy_x_HomeCharging` represents the stronger purchase pattern when subsidy and home charging are available together.
- `Income_x_Concern` combines financial ability with EV motivation.
- `Subsidy_x_Income` combines affordability with the economic incentive.
- `Low_Range_Anxiety` gives linear models an explicit indicator for the dominant low-anxiety group.

We are deliberately **not multiplying every feature by every other feature**. The goal is useful signal, not feature explosion.

## Nominal Categorical Encoding

In [15]:
categorical_cols = X_train.select_dtypes(include="object").columns

categorical_cols

Index(['Gender', 'City_Type', 'Current_Car_Type'], dtype='object')

In [16]:
X_train = pd.get_dummies(
    X_train ,
    columns=categorical_cols ,
    drop_first=True ,
    dtype=int
)

X_test = pd.get_dummies(
    X_test ,
    columns=categorical_cols ,
    drop_first=True ,
    dtype=int
)

X_test = X_test.reindex(columns=X_train.columns , fill_value=0)

X_train.shape , X_test.shape

((668665, 25), (286571, 25))

Good — the remaining nominal categories are now ready for standard machine-learning models. ✅

Only `Gender`, `City_Type`, and `Current_Car_Type` needed one-hot encoding. They do **not** have a natural order, so assigning values like `Urban=1`, `Suburban=2`, `Rural=3` would create a fake numerical relationship.

`drop_first=True` removes one redundant dummy column from each category group, and `reindex()` guarantees that Kaggle train and test have exactly the same final feature columns.

## Final Feature Check

In [17]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain missing values:", X_train.isnull().sum().sum())
print("Test missing values:", X_test.isnull().sum().sum())

print("\nNon-numerical train columns:", len(X_train.select_dtypes(exclude=np.number).columns))
print("Non-numerical test columns:", len(X_test.select_dtypes(exclude=np.number).columns))

print("\nTrain/Test columns identical:", X_train.columns.equals(X_test.columns))
print("ID inside model features:", "id" in X_train.columns)
print("Target inside model features:", "Will_Buy_EV" in X_train.columns)

Train shape: (668665, 25)
Test shape: (286571, 25)

Train missing values: 0
Test missing values: 0

Non-numerical train columns: 0
Non-numerical test columns: 0

Train/Test columns identical: True
ID inside model features: False
Target inside model features: False


In [18]:
X_train.head()

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,...,Subsidy_x_HomeCharging,Income_x_Concern,Subsidy_x_Income,Gender_Male,Gender_Other,City_Type_Suburban,City_Type_Urban,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,66,92887.0,23.4,2,3,7,1,1,0,0,...,0,9.2887,0.0000,1,0,1,0,0,1,0
1,38,30000.0,5.0,1,2,2,4,1,0,0,...,0,12.0000,0.0000,1,0,0,0,1,0,0
2,26,94389.0,36.8,1,8,15,5,0,1,0,...,0,47.1945,9.4389,0,0,0,1,0,1,0
3,66,73580.0,23.7,2,6,9,3,1,0,0,...,0,22.0740,0.0000,1,0,1,0,0,0,0
4,54,57898.0,50.8,1,2,3,3,1,0,0,...,0,17.3694,0.0000,1,0,1,0,0,0,0


In [19]:
X_train.dtypes.value_counts()

int64      20
float64     5
Name: count, dtype: int64

Nice — this is exactly what we wanted. ✅

At this point the model matrix should satisfy all of the important conditions:

**no missing values → no object columns → no identifier leakage → no target leakage → meaningful engineered features → identical train/test columns**

We have also kept the original Kaggle test IDs separately, so submission generation in Notebook 03 will remain straightforward.

## Save Processed Data

In [20]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True , exist_ok=True)

X_train.to_csv(processed_path / "X_train.csv" , index=False)
X_test.to_csv(processed_path / "X_test.csv" , index=False)

y.rename("Will_Buy_EV").to_csv(processed_path / "y_train.csv" , index=False)

train_ids.rename("id").to_csv(processed_path / "train_ids.csv" , index=False)
test_ids.rename("id").to_csv(processed_path / "test_ids.csv" , index=False)

list(processed_path.iterdir())

[WindowsPath('../data/processed/test_ids.csv'),
 WindowsPath('../data/processed/train_ids.csv'),
 WindowsPath('../data/processed/X_test.csv'),
 WindowsPath('../data/processed/X_train.csv'),
 WindowsPath('../data/processed/y_train.csv')]

In [21]:
saved_files = {
    file.name: round(file.stat().st_size / (1024 ** 2) , 2)
    for file in processed_path.iterdir()
}

pd.Series(saved_files , name="Size_MB").sort_index()

X_test.csv       21.63
X_train.csv      50.46
test_ids.csv      2.19
train_ids.csv     5.00
y_train.csv       1.91
Name: Size_MB, dtype: float64

Done — the processed model data is now saved separately. 🚀

Notebook 03 can load these files directly without repeating EDA or feature engineering:

- `X_train.csv` → processed competition training features
- `y_train.csv` → binary target
- `X_test.csv` → processed Kaggle test features
- `train_ids.csv` → original training IDs
- `test_ids.csv` → IDs needed for the final Kaggle submission

The **validation split belongs in Model Training**, not here.

## Feature Engineering Summary

| Task | Status |
|---|:---:|
| Raw train/test loaded independently | ✅ |
| Target encoded (`No=0`, `Yes=1`) | ✅ |
| `id` removed from model features | ✅ |
| Missing-value imputation | Not needed |
| Duplicate removal | Not needed |
| Binary variables encoded | ✅ |
| Range anxiety encoded ordinally | ✅ |
| Environmental concern order preserved | ✅ |
| Total charging infrastructure feature created | ✅ |
| Affordability / ownership feature created | ✅ |
| EDA-supported interaction features created | ✅ |
| Nominal categorical features one-hot encoded | ✅ |
| Train/test columns aligned | ✅ |
| All final model features numerical | ✅ |
| Processed files saved for Notebook 03 | ✅ |

Feature Engineering is complete. ⚡

The next notebook can now focus entirely on **model comparison, validation, tuning, blending, and Kaggle submission** without touching the raw preprocessing logic again.